# B2.13 · Attesting control intent for agents and MCP servers

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.12 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**.

| | |
|---|---|
| Tools used | in-toto, Sigstore, OSCAL, OPA, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Run the control-intent analyser over ten real agent and MCP repositories and read the attestation it produces for each.

**Why a security engineer needs it.** Control claims are asserted in a spreadsheet and never bound to a deployment. Nobody can say which repo, image, role, identity, gateway and guardrail the claim was about, so it cannot be re-checked when any of them change. The control it builds is: eleven skills scoped to one deployment_id, emitting an in-toto/DSSE attestation whose predicate carries per-control verdicts, evidence URIs, framework mappings and drift — with sandbox-egress and injection-screening capped at PARTIAL because their claims are not provable.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"We enforce least privilege" is true of some deployment, at some time, and nothing binds it to the system running right now. An attestation is the binding — and the discipline is that it must refuse to say more than it can show.

> **At CyberTravels.** “CyberTravels enforces least privilege” is true of some deployment at some time. An attestation is what binds it to the one running now — and refuses to claim more than it can show.

## 2 · The framework

```
   claim                          attestation
   "we enforce least privilege"   subject: deployment_id @ digest
            |                     predicate: per-control verdicts + evidence
        unbound                   signed, re-issuable on any change

   the ceiling that keeps it honest
   +------------------------------------------------+
   | INTENT_EVIDENCED   strongest static verdict     |
   | PARTIAL (capped)   sandbox egress, injection    |
   | PASS               not in the vocabulary        |
   +------------------------------------------------+
```

Every control claim in this pipeline has the same weakness: it is a sentence in
a document, and nothing binds it to a running system. "We enforce least
privilege" is true of some deployment, at some time, and there is no way to
re-check it when the image, the role or the tool surface changes.

An **attestation** fixes the binding. It is a signed statement about a specific
deployment, carrying per-control verdicts and the evidence behind each one, that
can be re-issued whenever anything it describes changes.

Two design decisions carry the whole idea.

**A `deployment_id` is the join key.** It resolves to a manifest of
content-addressed artefacts — repo at a commit, image by digest, IAM role,
workload identity, gateway route, guardrail ID, downstream services. Without it
an IAM finding, an identity entry and a gateway policy are three unrelated facts
about three things that may not be the same system.

**Eleven skills, not one.** One resolver, nine collectors split along
evidence-source boundaries (code, IAM, network, sandbox, identity, gateway,
ingestion, risk register, entitlements), and one signer. They split there
because each needs different API clients — and they stay separate because a
single mega-skill produces context bloat and verdicts nobody can read.

Then the part that makes it honest. **Not every control is equally verifiable:**

| Control | Confidence | Why |
|---|---|---|
| C1 default-deny / least privilege | HIGH | policy documents plus observed usage are readable |
| C3 identity chain / OBO | HIGH | delegation is impossible without an actor token, so its presence is proof |
| C4 gateway routing | HIGH **if** egress is enforced below the application | otherwise an agent opens a socket and bypasses it |
| C2 sandbox / no egress | **PARTIAL, capped** | absence of a covert channel is not provable; DNS and object-storage bypasses are documented |
| C5 injection screening | **PARTIAL, capped** | detector presence is verifiable; adaptive attacks drive published defences back above 95% success |

A tool that reports PASS on C2 or C5 is wrong, and the cap belongs in the
artefact rather than in a footnote.

## 3 · The skill that does the static half

### The skill — [`skills/attestation/agent-code-surface-analyzer/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/agent-code-surface-analyzer/SKILL.md)

```yaml
name: agent-code-surface-analyzer
description: >-
  Statically enumerate an agent or MCP server's declared tools, dangerous
  actions and declared-versus-actual capability surface from its repository
  or image. Use to inventory what a deployment can do before trusting what
  it says it does, to check MCP tool annotations against the code, or to
  baseline tool descriptions for rug-pull detection.
allowed-tools: Read, Grep, Glob, Bash
```

# Agent Code Surface Analyzer

**Controls:** Static basis for controls 1, 2 and 5

## What this establishes

Every other control is about constraining capability. This skill establishes
what the capability actually **is**, from the code rather than from the
manifest — because the manifest is a claim made by the thing being audited.

## Procedure

1. **Enumerate declared tools.** From the MCP tool manifest, the tool registry,
   or the decorator/registration sites in code.

2. **Classify each tool by what it actually reaches**, by locating sinks:
   - `subprocess`, `os.system`, `exec`, `eval` → **process execution**
   - `open(...,'w')`, file writes, `shutil`, `os.remove` → **filesystem write**
   - HTTP clients, sockets → **network egress**
   - SDK credential reads, environment access → **credential access**
   - `DELETE`, `DROP`, `TRUNCATE` → **destructive downstream**

3. **Cross-check annotations against the code.** MCP tool annotations
   (`readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint`) are
   **hints, not guarantees** — the specification says so explicitly, and a
   server can mislabel a destructive tool as read-only. Treat every annotation
   as a claim to verify, never as truth.

   Apply the spec's pessimistic defaults: an **unannotated** tool is assumed
   `destructiveHint: true` and `openWorldHint: true`.

4. **Record the declared-versus-actual delta.** A tool annotated `readOnlyHint`
   whose body writes a file is the highest-value finding this skill produces.

5. **Hash the tool descriptions.** Store `tool_description_hash` per tool. This
   is the rug-pull baseline: a server can change a tool's description after the
   client approved it, and the hash is what detects that.

## Output contract

```json
{
  "deployment_id": "str",
  "tools": [
    {"name": "str", "capability_class": ["process|filesystem|network|credential|destructive"],
     "annotations": {"readOnlyHint": false, "destructiveHint": true,
                     "idempotentHint": false, "openWorldHint": true},
     "annotations_present": true,
     "declared_vs_actual": "match|understated|overstated",
     "evidence": [{"file": "str", "line": 0, "sink": "str"}],
     "tool_description_hash": "str"}
  ],
  "dangerous_actions": ["str"],
  "unannotated_count": 0,
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Believing the annotations.** They are advisory. Cross-check or do not
  report on them at all.
- **Missing dynamic registration.** Tools registered at runtime from config do
  not appear in a static scan; record that as a `PARTIAL`, not a clean pass.
- **Hashing the tool name instead of the description.** The description is what
  the model reads and what a rug-pull changes.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/agent-code-surface-analyzer/scripts/agent_code_surface_analyzer.py
SCRIPT = "skills/attestation/agent-code-surface-analyzer/scripts/agent_code_surface_analyzer.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · Control intent, and why it is the honest static claim

Static analysis cannot show that a control **holds**. It can show that somebody **intended** it — an imported sandbox, a validated audience claim, a provenance tag. That is a smaller claim and a true one, so the analyser never emits PASS.

## 5 · Run against ten real repositories

These are the verdicts the analyser in `labs/attestation/control_intent.py` produced against the five most-deployed open-source MCP repositories and five most-used agent frameworks, cloned at HEAD. Not a simulation — the counts below are what the scan returned.

## 6 · What the scan actually found

Three findings worth more than the table.

## 7 · The artefact

An in-toto statement, subject-bound to the deployment, predicate in assessment-results vocabulary. The signer is a separate skill and a separate role — an attester that also decides whether it passed is not an attestation.

## What you just proved

Five controls resolve to INTENT_EVIDENCED, PARTIAL or NO_INTENT_FOUND and never to PASS. Across ten real repositories and fifty control evaluations the analyser returns 30 INTENT_EVIDENCED, 16 PARTIAL, 4 NO_INTENT_FOUND and zero PASS — with one widely-deployed MCP server shipping no tool annotations at all, so all of its tool sites inherit the specification's destructive, open-world default.

## Your turn

Run the analyser against one agent or MCP server you actually deploy. The interesting output is not the verdicts — it is the controls that come back NO_INTENT_FOUND, because those are the ones nobody has started.

---

**Next → [B2.14 · Bonus — Google Mantis, the pipeline in production](https://spbreed.github.io/cyber-commons/lessons/B2.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*